# 00 Run pipeline

Rebuilds every layer from `samples.tpch` and checks the result.

| Step | What it does | Output |
|---|---|---|
| Bronze | Copies the 8 source tables as is, plus `_ingested_at` | `workspace.bronze.*` |
| Silver | Typed 3NF tables with enforced quality, rejected rows go to quarantine | `workspace.silver.*`|
| Gold | Star schema and headline metrics for the CTO questions | `workspace.gold.*`|
| Validation | Runs all data-quality rules and shows pass/fail | results table below |

Names (catalog, source, as-of date) come from `src/tpch/config.py` and can be overridden with the `TPCH_CATALOG`, `TPCH_SOURCE` and `TPCH_AS_OF_DATE` environment variables.

In [0]:
%load_ext autoreload
%autoreload 2

## Run the pipeline

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))


from tpch import bronze, silver, gold, validation
# bronze.run(spark)
# silver.run(spark)
# gold.run(spark)

## Validate
Every rule returns the number of violating rows. 0 means pass. See `src/tpch/validation.py`.

In [0]:
display(validation.run(spark))

## Anomaly Detection & Alerts
Monitors operational and financial metrics for unexpected drops or anomalies, specifically checking week-over-week (WoW) net revenue decreases exceeding 20 percent. See `src/tpch/alerts.py`.

In [0]:
from tpch import alerts
alerts.run(spark)

## Master Pipeline Orchestrator
Sequentially executes the end-to-end Lakehouse workflow (Bronze ingestion, Silver 3NF transformation, Gold data mart aggregation, data quality validation, and anomaly monitoring) with built-in error handling. See `src/tpch/orchestrator.py`.

In [0]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from tpch import orchestrator

orchestrator.run_full_pipeline(spark, run_bronze=True)